# ============================================================
# MASTER THESIS TOPIC MODELING PIPELINE
# BERTopic + Sentiment + Emotion + Visualisations + Statistics
# ============================================================


In [7]:
# ============================================================
# 1) SETUP + LOAD
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import re
import ast
import random
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except ImportError:
    torch = None

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

project_dir = Path("/content/drive/MyDrive/Master Thesis")
file_path = project_dir / "emotionalabuse_threads_hot.csv"

results_dir = project_dir / "Results"
figures_dir = results_dir / "Figures"
tables_dir = results_dir / "Tables"
models_dir = results_dir / "Models"

for folder in [results_dir, figures_dir, tables_dir, models_dir]:
    folder.mkdir(parents=True, exist_ok=True)

out_dir = str(results_dir)

df = pd.read_csv(file_path, encoding="latin-1")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Results:", results_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (1002, 7)
Columns: ['date_utc', 'timestamp', 'title', 'text', 'subreddit', 'comments', 'url']
Results: /content/drive/MyDrive/Master Thesis/Results


In [2]:
# ============================================================
# 2) BUILD DOCUMENTS + MINIMAL CLEANING
# ============================================================

df["title"] = df["title"].fillna("").astype(str)
df["text"] = df["text"].fillna("").astype(str)

df["document"] = df["title"].str.strip() + "\n" + df["text"].str.strip()
df = df[df["document"].str.strip() != ""]

def clean_minimal(s: str) -> str:
    s = s.replace("[removed]", "").replace("[deleted]", "")
    s = re.sub(r"http\S+|www\.\S+", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["document"] = df["document"].astype(str).apply(clean_minimal)
df = df[df["document"].str.len() >= 50].reset_index(drop=True)

documents = df["document"].tolist()

print("Final number of documents:", len(documents))
print("Date range:", df["date_utc"].min(), "to", df["date_utc"].max())


Final number of documents: 988
Date range: 2025-03-28 to 2026-03-02


In [3]:
# ============================================================
# 3) DESCRIPTIVE STATISTICS FOR FINAL CORPUS
# ============================================================

# Word count per cleaned post/document
df["word_count"] = df["document"].apply(lambda x: len(str(x).split()))

# Character count per cleaned post/document
df["character_count"] = df["document"].apply(lambda x: len(str(x)))

# Helper function for mean, SD, min, max
def descriptive_stats(series):
    return pd.Series({
        "N": series.dropna().shape[0],
        "M": series.mean(),
        "SD": series.std(),
        "Min": series.min(),
        "Max": series.max()
    })

# Basic corpus descriptives
corpus_descriptives = pd.DataFrame({
    "word_count": descriptive_stats(df["word_count"]),
    "character_count": descriptive_stats(df["character_count"])
}).T

# Add available engagement variables if present
possible_engagement_cols = ["comments", "score", "ups", "likes", "num_comments"]

for col in possible_engagement_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        corpus_descriptives.loc[col] = descriptive_stats(df[col])

# Round values for reporting
corpus_descriptives_rounded = corpus_descriptives.round(2)

print(corpus_descriptives_rounded)

# Export table
corpus_descriptives_rounded.to_csv(tables_dir / "corpus_descriptive_statistics.csv")
corpus_descriptives_rounded.to_excel(tables_dir / "corpus_descriptive_statistics.xlsx")

print("Corpus descriptive statistics saved to:", tables_dir)


                     N        M       SD   Min      Max
word_count       988.0   379.67   412.21   4.0   4199.0
character_count  988.0  1992.72  2174.18  50.0  20753.0
comments         988.0     8.36    14.15   1.0    165.0
Corpus descriptive statistics saved to: /content/drive/MyDrive/Master Thesis/Results/Tables


In [4]:
# ============================================================
# 4) INSTALL DEPENDENCIES
# ============================================================

!pip -q install bertopic sentence-transformers umap-learn hdbscan scikit-learn gensim transformers accelerate plotly kaleido openpyxl scipy statsmodels


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.2 MB/s eta 0:00:00


In [5]:
# ============================================================
# 5) BERTOPIC CONFIG
# ============================================================

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

embedding_model = SentenceTransformer("all-MiniLM-L12-v2")

umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED
)

hdbscan_model = HDBSCAN(
    min_cluster_size=35,
    min_samples=2,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

n_docs = len(documents)

min_df_value = 2 if n_docs < 500 else 5
max_df_value = 0.9 if n_docs < 500 else 0.7

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=min_df_value,
    max_df=max_df_value
)

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = KeyBERTInspired()

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    top_n_words=10,
    calculate_probabilities=True,
    verbose=True
)

print("BERTopic configured.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BERTopic configured.


In [6]:
# ============================================================
# 6) FIT TOPIC MODEL
# ============================================================

topics, probs = topic_model.fit_transform(documents)

topics_red = topics
probs_red = probs

topic_info = topic_model.get_topic_info()

print(topic_info.head(20))

n_outliers = int(np.sum(np.array(topics_red) == -1))
n_topics = len([t for t in topic_info["Topic"].tolist() if t != -1])

print("\nNumber of topics:", n_topics)
print("Outliers:", n_outliers, "/", len(topics_red), f"({n_outliers / len(topics_red):.1%})")

model_path = models_dir / "bertopic_model"

topic_model.save(
    str(model_path),
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=False
)

print("Model saved to:", model_path)

2026-06-11 10:35:32,852 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

2026-06-11 10:37:03,768 - BERTopic - Embedding - Completed ✓
2026-06-11 10:37:03,773 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-11 10:37:14,236 - BERTopic - Dimensionality - Completed ✓
2026-06-11 10:37:14,237 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-11 10:37:14,293 - BERTopic - Cluster - Completed ✓
2026-06-11 10:37:14,298 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-11 10:37:17,411 - BERTopic - Representation - Completed ✓
2026-06-11 10:37:17,661 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


   Topic  Count                                               Name  \
0     -1    196    -1_sexual abuse_traumatic_traumatized_feel safe   
1      0    206  0_leaving abusive_physical abuse_need advice_f...   
2      1    167             1_treatment_hes trying_affection_yells   
3      2    116  2_abusive relationships_domestic abuse_physica...   
4      3     80  3_traumatized_traumas_accused cheating_pulled ...   
5      4     80  4_abuse im_relationship started_relationship y...   
6      5     67   5_left abusive_abuse physical_met ex_feel broken   
7      6     41  6_ex husband_past trauma_assaulted_sexual assault   
8      7     35  7_extremely abusive_verbally abuse_called cops...   

                                      Representation  \
0  [sexual abuse, traumatic, traumatized, feel sa...   
1  [leaving abusive, physical abuse, need advice,...   
2  [treatment, hes trying, affection, yells, punc...   
3  [abusive relationships, domestic abuse, physic...   
4  [traumatized, tr

In [8]:
# ============================================================
# 7) DIAGNOSTICS + COHERENCE
# ============================================================

from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

topics_array = np.array(topics_red)

n_outliers = int(np.sum(topics_array == -1))
n_total = len(topics_array)
outlier_share = n_outliers / n_total

print("Outliers (-1):", n_outliers, "/", n_total, f"({outlier_share:.1%})")

token_pattern = re.compile(r"\b[a-zA-Z]{2,}\b")

tokenized_docs = [
    token_pattern.findall(doc.lower())
    for doc in documents
]

dictionary = Dictionary(tokenized_docs)

topic_ids = [
    t for t in topic_model.get_topic_info()["Topic"].tolist()
    if t != -1
]

topic_words = []

for topic_id in topic_ids:
    top_terms = topic_model.get_topic(topic_id)
    if top_terms:
        topic_words.append([word for word, _ in top_terms[:10]])

if len(topic_words) >= 2:
    coherence_cv = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v"
    ).get_coherence()

    coherence_npmi = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_npmi"
    ).get_coherence()

    print("Coherence C_v:", round(coherence_cv, 3))
    print("Coherence NPMI:", round(coherence_npmi, 3))
else:
    coherence_cv = np.nan
    coherence_npmi = np.nan
    print("Coherence skipped.")

diagnostics_df = pd.DataFrame({
    "Metric": [
        "Number of documents",
        "Number of topics excluding outliers",
        "Number of outliers",
        "Outlier share",
        "Coherence C_v",
        "Coherence NPMI"
    ],
    "Value": [
        n_total,
        len(topic_ids),
        n_outliers,
        round(outlier_share, 4),
        round(coherence_cv, 4) if not np.isnan(coherence_cv) else np.nan,
        round(coherence_npmi, 4) if not np.isnan(coherence_npmi) else np.nan
    ]
})

diagnostics_df.to_csv(tables_dir / "topic_model_diagnostics.csv", index=False)
diagnostics_df.to_excel(tables_dir / "topic_model_diagnostics.xlsx", index=False)

diagnostics_df


Outliers (-1): 196 / 988 (19.8%)
Coherence C_v: 0.36
Coherence NPMI: -0.372


,Metric,Value
0,Number of documents,988.0000
1,Number of topics excluding outliers,8.0000
2,Number of outliers,196.0000
3,Outlier share,0.1984
4,Coherence C_v,0.3602
5,Coherence NPMI,-0.3725


In [9]:
# ============================================================
# 8) DOCUMENT-LEVEL ASSIGNMENT TABLE
# ============================================================

doc_info = pd.DataFrame({
    "Document_ID": range(len(documents)),
    "Document": documents,
    "Topic": topics_red
})

if probs_red is not None:
    probs_arr = np.asarray(probs_red)

    if probs_arr.ndim == 2:
        doc_info["Probability"] = probs_arr.max(axis=1)
    elif probs_arr.ndim == 1:
        doc_info["Probability"] = probs_arr
    else:
        doc_info["Probability"] = np.nan
else:
    doc_info["Probability"] = np.nan

metadata_cols = [
    col for col in ["date_utc", "timestamp", "title", "text", "subreddit", "url"]
    if col in df.columns
]

metadata_df = df[metadata_cols].reset_index(drop=True)

doc_info = pd.concat(
    [doc_info.reset_index(drop=True), metadata_df],
    axis=1
)

doc_info = doc_info.sort_values(
    ["Topic", "Probability"],
    ascending=[True, False]
).reset_index(drop=True)

doc_info.head()



,Document_ID,Document,Topic,Probability,date_utc,timestamp,title,text,subreddit,url
0,68.0,My place is finally clean. Its been 3 months s...,-1.0,0.525304,2026-02-03,1770134552,My place is finally clean.,Its been 3 months since I dumped his ass.I cou...,abusiverelationships,https://www.reddit.com/r/abusiverelationships/...
1,930.0,I thought I was done & ): Im feeling like Im g...,-1.0,0.355323,2026-03-02,1772421936,I feel like the same thoughts are coming back ...,I had a very toxic relationship before my curr...,abusiverelationships,https://www.reddit.com/r/abusiverelationships/...
2,708.0,A poem Emotional abuse makes me feel like a ra...,-1.0,0.340390,2026-02-25,1772006038,Im not crazy. (Long),"My first relationship was in a year long, and ...",abusiverelationships,https://www.reddit.com/r/abusiverelationships/...
3,755.0,An Ongoing Pattern of Betrayal and Intermitten...,-1.0,0.336861,2026-02-23,1771888460,How do i help my friend in another state get o...,My friend is stuck in an abusive relationship....,abusiverelationships,https://www.reddit.com/r/abusiverelationships/...
4,173.0,Managing ANGERRRRRRR after leaving After leavi...,-1.0,0.335605,2026-02-05,1770313286,Managing ANGERRRRRRR after leaving,After leaving my abuser. I 26F feel soo much a...,abusiverelationships,https://www.reddit.com/r/abusiverelationships/...


In [10]:
# ============================================================
# 9) EXPORT TEMPLATE FOR MANUAL TOPIC LABELS
# ============================================================

labels_df = topic_model.get_topic_info()
labels_df = labels_df[labels_df["Topic"] != -1][["Topic", "Count", "Name", "Representation"]].copy()

def repr_to_str(x):
    if isinstance(x, list):
        return " | ".join(map(str, x))
    try:
        as_list = ast.literal_eval(x)
        if isinstance(as_list, list):
            return " | ".join(map(str, as_list))
    except Exception:
        pass
    return str(x)

labels_df["Representation"] = labels_df["Representation"].apply(repr_to_str)
labels_df["Topic_Label"] = ""

labels_path = tables_dir / "topic_labels_to_fill.xlsx"
labels_df.to_excel(labels_path, index=False)

print("Saved label template:", labels_path)
labels_df.head()

Saved label template: /content/drive/MyDrive/Master Thesis/Results/Tables/topic_labels_to_fill.xlsx


,Topic,Count,Name,Representation,Topic_Label
1,0,206,0_leaving abusive_physical abuse_need advice_f...,leaving abusive | physical abuse | need advice...,
2,1,167,1_treatment_hes trying_affection_yells,treatment | hes trying | affection | yells | p...,
3,2,116,2_abusive relationships_domestic abuse_physica...,abusive relationships | domestic abuse | physi...,
4,3,80,3_traumatized_traumas_accused cheating_pulled ...,traumatized | traumas | accused cheating | pul...,
5,4,80,4_abuse im_relationship started_relationship y...,abuse im | relationship started | relationship...,


In [ ]:
# ============================================================
# 10) SENTIMENT ANALYSIS
# ============================================================

from transformers import pipeline
from tqdm.auto import tqdm

rep_df = doc_info.copy()

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    device=0,
    top_k=None
)

def batched(lst, batch_size=16):
    for i in range(0, len(lst), batch_size):
        yield lst[i:i + batch_size]

texts = rep_df["Document"].tolist()

rows = []
label_mode = None

for batch in tqdm(batched(texts, 16), total=(len(texts) + 15) // 16):
    out = sentiment_pipe(batch, truncation=True, max_length=512)

    for doc_out in out:
        doc_scores = [doc_out] if isinstance(doc_out, dict) else doc_out
        score_map = {d["label"]: float(d["score"]) for d in doc_scores}

        if label_mode is None:
            if all(k in score_map for k in ["LABEL_0", "LABEL_1", "LABEL_2"]):
                label_mode = "LABEL"
            elif all(k in score_map for k in ["negative", "neutral", "positive"]):
                label_mode = "WORD"
            else:
                label_mode = "UNKNOWN"

            print("Detected sentiment label scheme:", label_mode)

        if label_mode == "LABEL":
            neg = score_map.get("LABEL_0", np.nan)
            neu = score_map.get("LABEL_1", np.nan)
            pos = score_map.get("LABEL_2", np.nan)
        elif label_mode == "WORD":
            neg = score_map.get("negative", np.nan)
            neu = score_map.get("neutral", np.nan)
            pos = score_map.get("positive", np.nan)
        else:
            neg = neu = pos = np.nan

        rows.append({
            "sent_neg": neg,
            "sent_neu": neu,
            "sent_pos": pos
        })

sent_scores_df = pd.DataFrame(rows)

rep_df[["sent_neg", "sent_neu", "sent_pos"]] = sent_scores_df[
    ["sent_neg", "sent_neu", "sent_pos"]
].values

best = rep_df[["sent_neg", "sent_neu", "sent_pos"]].idxmax(axis=1)

rep_df["sentiment_label"] = best.map({
    "sent_neg": "negative",
    "sent_neu": "neutral",
    "sent_pos": "positive"
})

print("Sentiment analysis completed.")
rep_df.head()



config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

  0%|          | 0/63 [00:00<?, ?it/s]

Detected sentiment label scheme: LABEL


In [ ]:
# ============================================================
# 11) EMOTION MULTILABEL ANALYSIS
# ============================================================

emotion_pipe = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-emotion-multilabel-latest",
    top_k=None,
    device=0
)

emotion_texts = rep_df["Document"].tolist()

all_emotion_rows = []

for batch in tqdm(batched(emotion_texts, 8), total=(len(emotion_texts) + 7) // 8):
    out = emotion_pipe(batch, truncation=True, max_length=512)

    for doc_scores in out:
        row = {d["label"]: float(d["score"]) for d in doc_scores}
        all_emotion_rows.append(row)

emotion_df = pd.DataFrame(all_emotion_rows).fillna(0.0)
emotion_cols = list(emotion_df.columns)

rep_df = pd.concat(
    [rep_df.reset_index(drop=True), emotion_df.reset_index(drop=True)],
    axis=1
)

print("Emotion analysis completed.")
rep_df.head()


In [ ]:
# ============================================================
# 12) TOPIC-LEVEL AGGREGATION
# ============================================================

sent_dist = (
    rep_df.groupby(["Topic", "sentiment_label"])
    .size()
    .unstack(fill_value=0)
)

sent_dist = sent_dist.div(sent_dist.sum(axis=1), axis=0).reset_index()

emo_means = rep_df.groupby("Topic")[emotion_cols].mean().reset_index()

sent_means = rep_df.groupby("Topic")[["sent_neg", "sent_neu", "sent_pos"]].mean().reset_index()

topic_summary = (
    sent_dist
    .merge(emo_means, on="Topic", how="inner")
    .merge(sent_means, on="Topic", how="left")
)

topic_info_final = topic_model.get_topic_info()
topic_info_final = topic_info_final[topic_info_final["Topic"] != -1].copy()

final = topic_info_final.merge(topic_summary, on="Topic", how="left")

final.head()



In [ ]:
# ============================================================
# 13) EXAMPLE POSTS
# ============================================================

EXAMPLES_PER_TOPIC = 3

examples = (
    rep_df.sort_values(["Topic", "Probability"], ascending=[True, False])
    .groupby("Topic")
    .head(EXAMPLES_PER_TOPIC)
    [["Topic", "Probability", "sentiment_label", "Document"]]
).reset_index(drop=True)

examples.head()

In [ ]:
# ============================================================
# 14) BASELINE EXPORTS
# ============================================================

final.to_csv(tables_dir / "topic_summary_unlabeled.csv", index=False)
doc_info.to_csv(tables_dir / "documents_with_topics_unlabeled.csv", index=False)
rep_df.to_csv(tables_dir / "documents_with_sentiment_emotion_unlabeled.csv", index=False)
examples.to_csv(tables_dir / "topic_examples_unlabeled.csv", index=False)

final.to_excel(tables_dir / "topic_summary_unlabeled.xlsx", index=False)
doc_info.to_excel(tables_dir / "documents_with_topics_unlabeled.xlsx", index=False)
rep_df.to_excel(tables_dir / "documents_with_sentiment_emotion_unlabeled.xlsx", index=False)
examples.to_excel(tables_dir / "topic_examples_unlabeled.xlsx", index=False)

print("Baseline exports saved to:", tables_dir)


In [ ]:
# ============================================================
# 15) INTEGRATE MANUAL LABELS
# ============================================================

import pandas as pd
import numpy as np
import textwrap
from pathlib import Path

topic2label = {
    0: "Leaving Abusive Partners with Children and Custody Fears",
    1: "Trauma Bonding, Self-Blame, and Relationship Violence",
    2: "Recognizing and Interpreting Abuse Dynamics",
    3: "Healing and Emotional Recovery After Leaving Abuse",
    4: "Post-Separation Harassment and Social Invalidation",
    5: "Guilt, Self-Blame, and Emotional Coercion",
    6: "Trauma Bonding and Early Recovery After Abuse",
    7: "Emotional Blackmail and Psychological Manipulation"
}


labels_path = tables_dir / "topic_labels_to_fill.xlsx"

if labels_path.exists():
    labels_df = pd.read_excel(labels_path)
    labels_df.columns = labels_df.columns.str.strip()
    labels_df["Topic"] = pd.to_numeric(labels_df["Topic"], errors="coerce")

    print("Topics found in Excel label file:")
    print(sorted(labels_df["Topic"].dropna().astype(int).unique()))

    print("Topics found in BERTopic final table:")
    print(sorted(final["Topic"].dropna().astype(int).unique()))

    print(
        "\nCheck carefully: if Excel uses 1–8 but BERTopic uses 0–7, "
        "the labels will be shifted or missing."
    )


topic_model.set_topic_labels(topic2label)


from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import MDS
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import textwrap

def wrap_label(label, width=28):
    return "<br>".join(textwrap.wrap(str(label), width=width))


def get_valid_topic_embeddings(topic_model, valid_topics):
    """
    Extract topic embeddings in the same order as valid_topics.
    This avoids mismatches between BERTopic's internal topic order and topic IDs.
    """
    topic_info = topic_model.get_topic_info().copy()

    topic_info = topic_info[topic_info["Topic"].isin(valid_topics)].copy()
    topic_info = topic_info.sort_values("Topic")

    embeddings = topic_model.topic_embeddings_

    if len(embeddings) == len(topic_info):
        topic_info["Embedding_Index"] = range(len(topic_info))
    else:
        topic_info["Embedding_Index"] = topic_info["Topic"].astype(int)

    selected_embeddings = []

    for _, row in topic_info.iterrows():
        idx = int(row["Embedding_Index"])
        selected_embeddings.append(embeddings[idx])

    selected_embeddings = np.array(selected_embeddings)

    return topic_info["Topic"].astype(int).tolist(), selected_embeddings

for df_name in ["doc_info", "rep_df", "examples", "final"]:
    if df_name in globals():
        df = globals()[df_name]
        if "Topic" in df.columns:
            df["Topic_Label"] = df["Topic"].map(topic2label)
            df["Topic_Label"] = df["Topic_Label"].fillna("Outlier")

print("Manual topic labels integrated.")

display(
    final[["Topic", "Topic_Label", "Count"]]
    .sort_values("Topic")
)


In [ ]:
# ============================================================
# 16) STATISTICAL ANALYSES
# Chi-square test, ANOVAs, and Tukey post-hoc tests
# ============================================================

from scipy.stats import chi2_contingency, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import pandas as pd
import numpy as np



topic2label = {
    0: "Leaving Abusive Partners with Children and Custody Fears",
    1: "Trauma Bonding, Self-Blame, and Relationship Violence",
    2: "Recognizing and Interpreting Abuse Dynamics",
    3: "Healing and Emotional Recovery After Leaving Abuse",
    4: "Post-Separation Harassment and Social Invalidation",
    5: "Guilt, Self-Blame, and Emotional Coercion",
    6: "Trauma Bonding and Early Recovery After Abuse",
    7: "Emotional Blackmail and Psychological Manipulation"
}


analysis_df = rep_df[rep_df["Topic"] != -1].copy()

analysis_df["Topic"] = analysis_df["Topic"].astype(int)
analysis_df["Topic_Label"] = analysis_df["Topic"].map(topic2label)

print("Number of posts included in statistical analyses:", len(analysis_df))
print("Topics included:", sorted(analysis_df["Topic"].unique()))



possible_emotion_cols = [
    "anger", "anticipation", "disgust", "fear", "joy", "love",
    "optimism", "pessimism", "sadness", "surprise", "trust"
]

emotion_cols = [c for c in possible_emotion_cols if c in analysis_df.columns]

print("Emotion columns included in ANOVAs:")
print(emotion_cols)


# ============================================================
# Chi-square test: Sentiment distribution across topics
# ============================================================

sentiment_table = pd.crosstab(
    analysis_df["Topic_Label"],
    analysis_df["sentiment_label"]
)

print("\nSentiment contingency table:")
display(sentiment_table)

chi2, p, dof, expected = chi2_contingency(sentiment_table)

print("\nChi-square test of independence:")
print(f"χ²({dof}) = {chi2:.2f}, p = {p:.4f}")

expected_df = pd.DataFrame(
    expected,
    index=sentiment_table.index,
    columns=sentiment_table.columns
)

print("\nExpected frequencies:")
display(expected_df.round(2))


small_expected = (expected_df < 5).sum().sum()

print(f"\nNumber of expected cells below 5: {small_expected}")

if small_expected > 0:
    print(
        "Warning: Some expected frequencies are below 5. "
        "Interpret the chi-square result cautiously."
    )

chi_square_summary = pd.DataFrame({
    "Test": ["Chi-square test of independence"],
    "Variable_1": ["Topic"],
    "Variable_2": ["Sentiment label"],
    "Chi_square": [chi2],
    "df": [dof],
    "p": [p],
    "Expected_cells_below_5": [small_expected]
})

chi_square_summary["p_formatted"] = chi_square_summary["p"].apply(
    lambda x: "< .001" if x < .001 else f"{x:.3f}"
)


# ============================================================
# One-way ANOVAs: Emotion scores across topics
# ============================================================

anova_results = []

for emotion in emotion_cols:
    # Create one group per topic
    groups = [
        group[emotion].dropna()
        for _, group in analysis_df.groupby("Topic_Label")
    ]

    # Run one-way ANOVA
    f_stat, p_value = f_oneway(*groups)

    anova_results.append({
        "Emotion": emotion,
        "F": f_stat,
        "df_between": analysis_df["Topic_Label"].nunique() - 1,
        "df_within": len(analysis_df) - analysis_df["Topic_Label"].nunique(),
        "p": p_value
    })

anova_results_df = pd.DataFrame(anova_results)

# Add formatted p-values
anova_results_df["p_formatted"] = anova_results_df["p"].apply(
    lambda x: "< .001" if x < .001 else f"{x:.3f}"
)

# Sort by p-value
anova_results_df = anova_results_df.sort_values("p")

print("\nOne-way ANOVA results for emotion scores across topics:")
display(anova_results_df)


# ============================================================
# Tukey post-hoc tests for significant ANOVAs
# ============================================================

# Run Tukey tests only for emotions with significant ANOVA results
significant_emotions = anova_results_df.loc[
    anova_results_df["p"] < .05,
    "Emotion"
].tolist()

print("\nEmotions with significant ANOVA results:")
print(significant_emotions)

tukey_results = {}

for emotion in significant_emotions:
    tukey_data = analysis_df[["Topic_Label", emotion]].dropna().copy()

    tukey = pairwise_tukeyhsd(
        endog=tukey_data[emotion],
        groups=tukey_data["Topic_Label"],
        alpha=0.05
    )

    tukey_df = pd.DataFrame(
        data=tukey.summary().data[1:],
        columns=tukey.summary().data[0]
    )

    tukey_df.insert(0, "Emotion", emotion)

    tukey_results[emotion] = tukey_df

    print(f"\nTukey post-hoc results for {emotion}:")
    display(tukey_df)


# ============================================================
# Export statistical results
# ============================================================

chi_square_summary.to_excel(
    tables_dir / "chi_square_sentiment_summary.xlsx",
    index=False
)

sentiment_table.to_excel(
    tables_dir / "chi_square_sentiment_observed.xlsx"
)

expected_df.to_excel(
    tables_dir / "chi_square_sentiment_expected.xlsx"
)

anova_results_df.to_excel(
    tables_dir / "anova_results_emotions.xlsx",
    index=False
)

# Save Tukey results to separate Excel sheets
with pd.ExcelWriter(tables_dir / "tukey_posthoc_results.xlsx") as writer:
    for emotion, tukey_df in tukey_results.items():
        tukey_df.to_excel(
            writer,
            sheet_name=emotion[:31],
            index=False
        )

print("\nStatistical analysis files exported to:", tables_dir)
print("- chi_square_sentiment_summary.xlsx")
print("- chi_square_sentiment_observed.xlsx")
print("- chi_square_sentiment_expected.xlsx")
print("- anova_results_emotions.xlsx")
print("- tukey_posthoc_results.xlsx")

In [ ]:
# ============================================================
# 17) LABELED EXPORTS
# ============================================================

final.to_csv(tables_dir / "topic_summary_labeled.csv", index=False)
doc_info.to_csv(tables_dir / "documents_with_topics_labeled.csv", index=False)
rep_df.to_csv(tables_dir / "documents_with_sentiment_emotion_labeled.csv", index=False)
examples.to_csv(tables_dir / "topic_examples_labeled.csv", index=False)

final.to_excel(tables_dir / "topic_summary_labeled.xlsx", index=False)
doc_info.to_excel(tables_dir / "documents_with_topics_labeled.xlsx", index=False)
rep_df.to_excel(tables_dir / "documents_with_sentiment_emotion_labeled.xlsx", index=False)
examples.to_excel(tables_dir / "topic_examples_labeled.xlsx", index=False)

print("Labeled tables saved to:", tables_dir)


In [ ]:
# ============================================================
# 18) VISUALISATIONS
# ============================================================

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import textwrap
import numpy as np
import pandas as pd


PLOT_FONT = "Times New Roman, Times, serif"

pio.templates.default = "plotly_white"

def wrap_label(label, width=28):
    """Wrap long labels using HTML line breaks for Plotly."""
    return "<br>".join(textwrap.wrap(str(label), width=width))


def wrap_series(series, width=28):
    return series.astype(str).apply(lambda x: wrap_label(x, width=width))


def save_plotly(fig, filename, width=1800, height=1100):
    """Save interactive HTML and high-resolution PNG."""
    html_path = figures_dir / f"{filename}.html"
    png_path = figures_dir / f"{filename}.png"

    fig.write_html(
        str(html_path),
        include_plotlyjs=True,
        full_html=True
    )

    try:
        fig.write_image(
            str(png_path),
            width=width,
            height=height,
            scale=2
        )
        print("Saved PNG:", png_path)
    except Exception as e:
        print("PNG export failed for", filename)
        print("Reason:", e)
        print("HTML was still saved.")

    print("Saved HTML:", html_path)


def apply_thesis_style(
    fig,
    title_size=30,
    axis_title_size=24,
    tick_size=18,
    legend_size=18,
    title_x=0.5
):
    """Apply consistent thesis style."""
    fig.update_layout(
        font=dict(
            family=PLOT_FONT,
            size=tick_size,
            color="black"
        ),
        title=dict(
            font=dict(
                family=PLOT_FONT,
                size=title_size,
                color="black"
            ),
            x=title_x,
            xanchor="center"
        ),
        legend=dict(
            font=dict(
                family=PLOT_FONT,
                size=legend_size,
                color="black"
            ),
            title_font=dict(
                family=PLOT_FONT,
                size=legend_size,
                color="black"
            )
        ),
        paper_bgcolor="white",
        plot_bgcolor="white"
    )

    fig.update_xaxes(
        title_font=dict(family=PLOT_FONT, size=axis_title_size, color="black"),
        tickfont=dict(family=PLOT_FONT, size=tick_size, color="black"),
        showline=True,
        linewidth=1,
        linecolor="black",
        mirror=False,
        ticks="outside",
        gridcolor="rgba(0,0,0,0.10)"
    )

    fig.update_yaxes(
        title_font=dict(family=PLOT_FONT, size=axis_title_size, color="black"),
        tickfont=dict(family=PLOT_FONT, size=tick_size, color="black"),
        showline=True,
        linewidth=1,
        linecolor="black",
        mirror=False,
        ticks="outside",
        gridcolor="rgba(0,0,0,0.10)"
    )

    return fig


plot_df = final[final["Topic"] != -1].copy()
plot_df["Topic"] = plot_df["Topic"].astype(int)
plot_df = plot_df.sort_values("Topic")

plot_df["Topic_Label"] = plot_df["Topic"].map(topic2label)
plot_df["Topic_Label"] = plot_df["Topic_Label"].fillna("Outlier")
plot_df["Topic_Label_Wrapped"] = wrap_series(plot_df["Topic_Label"], width=32)

valid_topics = sorted(plot_df["Topic"].unique().tolist())
n_valid = len(valid_topics)

print("Topics included in figures:", valid_topics)
print(plot_df[["Topic", "Topic_Label", "Count"]])

# ============================================================
# Figure: Number of Posts per Topic
# ============================================================

count_plot = plot_df.sort_values("Count", ascending=True).copy()

fig = px.bar(
    count_plot,
    x="Count",
    y="Topic_Label_Wrapped",
    orientation="h",
    text="Count",
    labels={
        "Count": "Number of Posts",
        "Topic_Label_Wrapped": "Topic"
    },
    title="<b>Number of Posts per Topic</b>"
)

fig.update_traces(
    textposition="outside",
    textfont=dict(family=PLOT_FONT, size=18, color="black"),
    cliponaxis=False
)

fig.update_layout(
    width=1900,
    height=1000,
    margin=dict(l=420, r=160, t=130, b=100),
    showlegend=False
)

fig.update_xaxes(range=[0, count_plot["Count"].max() * 1.15])
fig.update_yaxes(categoryorder="array", categoryarray=count_plot["Topic_Label_Wrapped"])

fig = apply_thesis_style(fig)

save_plotly(
    fig,
    "topic_document_counts_labeled",
    width=1900,
    height=1000
)

# ============================================================
# Figure: Top c-TF-IDF Terms per Topic
# ============================================================

from plotly.subplots import make_subplots
import math

top_n_words = 5

topic_term_rows = []

for topic in valid_topics:
    topic_terms = topic_model.get_topic(topic)
    if topic_terms is None:
        continue

    for term, score in topic_terms[:top_n_words]:
        topic_term_rows.append({
            "Topic": topic,
            "Topic_Label": topic2label.get(topic, f"Topic {topic}"),
            "Term": term,
            "c-TF-IDF Score": score
        })

topic_terms_df = pd.DataFrame(topic_term_rows)

n_cols = 2
n_rows = math.ceil(n_valid / n_cols)

subplot_titles = [
    wrap_label(topic2label.get(topic, f"Topic {topic}"), width=34)
    for topic in valid_topics
]

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.18,
    vertical_spacing=0.16
)

for i, topic in enumerate(valid_topics):
    row = i // n_cols + 1
    col = i % n_cols + 1

    tmp = topic_terms_df[topic_terms_df["Topic"] == topic].copy()
    tmp = tmp.sort_values("c-TF-IDF Score", ascending=True)

    fig.add_trace(
        go.Bar(
            x=tmp["c-TF-IDF Score"],
            y=tmp["Term"],
            orientation="h",
            showlegend=False,
            hovertemplate=(
                "Topic: " + topic2label.get(topic, f"Topic {topic}") +
                "<br>Term: %{y}" +
                "<br>c-TF-IDF Score: %{x:.3f}<extra></extra>"
            )
        ),
        row=row,
        col=col
    )

fig.update_layout(
    title="<b>Top c-TF-IDF Terms per Topic</b>",
    width=1900,
    height=max(1200, 430 * n_rows),
    margin=dict(l=140, r=80, t=180, b=100)
)

for ann in fig.layout.annotations:
    ann.font = dict(family=PLOT_FONT, size=18, color="black")

fig = apply_thesis_style(
    fig,
    title_size=30,
    axis_title_size=20,
    tick_size=16,
    legend_size=16
)

save_plotly(
    fig,
    "bertopic_barchart_all_topics_labeled",
    width=1900,
    height=max(1200, 430 * n_rows)
)

# ============================================================
# BERTopic Intertopic Distance Map
# ============================================================

import copy
import textwrap

def wrap_label(label, width=28):
    return "<br>".join(textwrap.wrap(str(label), width=width))


def replace_topic_text(obj, topic2label):
    """
    Recursively replace 'Topic 0', 'Topic 1', etc. in Plotly figure objects.
    This helps patch BERTopic's built-in visualisation.
    """
    if isinstance(obj, dict):
        return {
            key: replace_topic_text(value, topic2label)
            for key, value in obj.items()
        }

    elif isinstance(obj, list):
        return [
            replace_topic_text(item, topic2label)
            for item in obj
        ]

    elif isinstance(obj, str):
        new_text = obj
        for topic, label in topic2label.items():
            new_text = new_text.replace(
                f"Topic {topic}",
                wrap_label(label, width=28)
            )
        return new_text

    else:
        return obj


if n_valid >= 2:

    topic_model.set_topic_labels(topic2label)

    fig = topic_model.visualize_topics(
        topics=valid_topics,
        custom_labels=True,
        title="<b>Intertopic Distance Map of Identified BERTopic Clusters</b>",
        width=1500,
        height=1000
    )


    fig_dict = fig.to_dict()
    fig_dict = replace_topic_text(fig_dict, topic2label)
    fig = go.Figure(fig_dict)

    fig.update_layout(
        width=1500,
        height=1000,
        margin=dict(l=100, r=100, t=140, b=140),
        font=dict(
            family="Times New Roman, Times, serif",
            size=16,
            color="black"
        ),
        title=dict(
            font=dict(
                family="Times New Roman, Times, serif",
                size=28,
                color="black"
            ),
            x=0.5,
            xanchor="center"
        ),
        paper_bgcolor="white",
        plot_bgcolor="white"
    )

    fig.update_xaxes(
        title_font=dict(
            family="Times New Roman, Times, serif",
            size=22,
            color="black"
        ),
        tickfont=dict(
            family="Times New Roman, Times, serif",
            size=16,
            color="black"
        )
    )

    fig.update_yaxes(
        title_font=dict(
            family="Times New Roman, Times, serif",
            size=22,
            color="black"
        ),
        tickfont=dict(
            family="Times New Roman, Times, serif",
            size=16,
            color="black"
        )
    )

    save_plotly(
        fig,
        "bertopic_intertopic_distance_labeled",
        width=1500,
        height=1000
    )

    print("BERTopic intertopic distance map saved with manual labels.")

# ============================================================
# Figure: Topic Similarity Matrix
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import plotly.express as px
import textwrap


def wrap_label(label, width=34):
    return "<br>".join(textwrap.wrap(str(label), width=width))


def get_valid_topic_embeddings(topic_model, valid_topics):
    """
    Extract topic embeddings in the same order as valid_topics.
    This prevents mismatches between BERTopic's internal order and topic IDs.
    """
    topic_info = topic_model.get_topic_info().copy()

    topic_info = topic_info[topic_info["Topic"].isin(valid_topics)].copy()
    topic_info = topic_info.sort_values("Topic")

    embeddings = topic_model.topic_embeddings_

    if len(embeddings) == len(topic_info):
        topic_info["Embedding_Index"] = range(len(topic_info))
    else:
        topic_info["Embedding_Index"] = topic_info["Topic"].astype(int)

    selected_embeddings = []

    for _, row in topic_info.iterrows():
        idx = int(row["Embedding_Index"])
        selected_embeddings.append(embeddings[idx])

    selected_embeddings = np.array(selected_embeddings)

    return topic_info["Topic"].astype(int).tolist(), selected_embeddings



if n_valid >= 2:

    ordered_topics, topic_embeddings_selected = get_valid_topic_embeddings(
        topic_model,
        valid_topics
    )

    similarity_matrix = cosine_similarity(topic_embeddings_selected)

    topic_labels_full = [
        topic2label.get(topic, f"Topic {topic}")
        for topic in ordered_topics
    ]

    topic_labels_wrapped = [
        wrap_label(label, width=34)
        for label in topic_labels_full
    ]

    similarity_df = pd.DataFrame(
        similarity_matrix,
        index=topic_labels_wrapped,
        columns=topic_labels_wrapped
    )


    similarity_df.to_csv(tables_dir / "topic_similarity_matrix_labeled.csv")
    similarity_df.to_excel(tables_dir / "topic_similarity_matrix_labeled.xlsx")


    fig = px.imshow(
        similarity_df,
        labels=dict(
            x="Topic",
            y="Topic",
            color="Similarity Score"
        ),
        title="<b>Topic Similarity Matrix</b>",
        aspect="auto",
        color_continuous_scale=[
            [0.0, "#f7fbff"],
            [0.2, "#deebf7"],
            [0.4, "#c6dbef"],
            [0.6, "#9ecae1"],
            [0.8, "#6baed6"],
            [1.0, "#2171b5"]
        ],
        zmin=0.5,
        zmax=1.0
    )

    fig.update_traces(
        hovertemplate=(
            "Topic X: %{x}<br>"
            "Topic Y: %{y}<br>"
            "Similarity Score: %{z:.3f}"
            "<extra></extra>"
        )
    )

    fig.update_layout(
        width=1700,
        height=1400,
        margin=dict(l=520, r=140, t=150, b=520),
        font=dict(
            family="Times New Roman, Times, serif",
            size=14,
            color="black"
        ),
        title=dict(
            font=dict(
                family="Times New Roman, Times, serif",
                size=30,
                color="black"
            ),
            x=0.5,
            xanchor="center"
        ),
        paper_bgcolor="white",
        plot_bgcolor="white"
    )

    fig.update_xaxes(
        title_font=dict(
            family="Times New Roman, Times, serif",
            size=24,
            color="black"
        ),
        tickfont=dict(
            family="Times New Roman, Times, serif",
            size=13,
            color="black"
        ),
        tickangle=45,
        automargin=True
    )

    fig.update_yaxes(
        title_font=dict(
            family="Times New Roman, Times, serif",
            size=24,
            color="black"
        ),
        tickfont=dict(
            family="Times New Roman, Times, serif",
            size=13,
            color="black"
        ),
        automargin=True
    )


    fig.update_layout(
        coloraxis_colorbar=dict(
            title=dict(
                text="Similarity Score",
                font=dict(
                    family="Times New Roman, Times, serif",
                    size=16,
                    color="black"
                )
            ),
            tickfont=dict(
                family="Times New Roman, Times, serif",
                size=14,
                color="black"
            )
        )
    )

    save_plotly(
        fig,
        "bertopic_similarity_matrix_labeled",
        width=1700,
        height=1400
    )

    print("Similarity matrix saved with full topic labels.")

# ============================================================
# Figure: Hierarchical Clustering
# ============================================================

if n_valid >= 2:
    import scipy.cluster.hierarchy as sch
    from scipy.spatial.distance import pdist

    topic_embeddings = topic_model.topic_embeddings_
    valid_embeddings = np.array([topic_embeddings[topic] for topic in valid_topics])

    labels_wrapped = [
        wrap_label(topic2label.get(topic, f"Topic {topic}"), width=38)
        for topic in valid_topics
    ]


    distances = pdist(valid_embeddings, metric="cosine")
    linkage = sch.linkage(distances, method="ward")

    dendro = sch.dendrogram(
        linkage,
        labels=labels_wrapped,
        orientation="left",
        no_plot=True
    )

    fig = go.Figure()

    for xs, ys in zip(dendro["dcoord"], dendro["icoord"]):
        fig.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode="lines",
                line=dict(width=2),
                showlegend=False,
                hoverinfo="skip"
            )
        )

    fig.update_layout(
        title="<b>Hierarchical Clustering of Topics</b>",
        width=1600,
        height=1000,
        margin=dict(l=520, r=100, t=140, b=100),
        yaxis=dict(
            tickmode="array",
            tickvals=[5 + 10 * i for i in range(len(dendro["ivl"]))],
            ticktext=dendro["ivl"]
        ),
        xaxis_title="Distance"
    )

    fig = apply_thesis_style(
        fig,
        title_size=30,
        axis_title_size=24,
        tick_size=16,
        legend_size=18
    )

    save_plotly(
        fig,
        "bertopic_hierarchy_labeled",
        width=1600,
        height=1000
    )

# ============================================================
# Figure: Term Score Decline per Topic
# ============================================================

term_rank_rows = []

for topic in valid_topics:
    terms = topic_model.get_topic(topic)
    if terms is None:
        continue

    for rank, (term, score) in enumerate(terms[:10], start=1):
        term_rank_rows.append({
            "Topic": topic,
            "Topic_Label": topic2label.get(topic, f"Topic {topic}"),
            "Term Rank": rank,
            "c-TF-IDF Score": score,
            "Term": term
        })

term_rank_df = pd.DataFrame(term_rank_rows)

fig = go.Figure()

for topic in valid_topics:
    tmp = term_rank_df[term_rank_df["Topic"] == topic].copy()

    fig.add_trace(
        go.Scatter(
            x=tmp["Term Rank"],
            y=tmp["c-TF-IDF Score"],
            mode="lines+markers",
            name=topic2label.get(topic, f"Topic {topic}"),
            hovertemplate=(
                "Topic: " + topic2label.get(topic, f"Topic {topic}") +
                "<br>Term Rank: %{x}" +
                "<br>c-TF-IDF Score: %{y:.3f}" +
                "<br>Term: %{customdata}<extra></extra>"
            ),
            customdata=tmp["Term"]
        )
    )

fig.update_layout(
    title="<b>Term Score Decline per Topic</b>",
    width=1600,
    height=1000,
    margin=dict(l=120, r=360, t=140, b=120),
    xaxis_title="Term Rank",
    yaxis_title="c-TF-IDF Score",
    legend_title="Topic"
)

fig.update_xaxes(dtick=1)

fig = apply_thesis_style(
    fig,
    title_size=30,
    axis_title_size=24,
    tick_size=18,
    legend_size=14
)

save_plotly(
    fig,
    "bertopic_term_rank_labeled",
    width=1600,
    height=1000
)

# ============================================================
# Figure: Mean Sentiment Probabilities per Topic
# ============================================================

sent_long = plot_df.melt(
    id_vars=["Topic", "Topic_Label", "Topic_Label_Wrapped"],
    value_vars=["sent_neg", "sent_neu", "sent_pos"],
    var_name="Sentiment",
    value_name="Mean Probability"
)

sent_long["Sentiment"] = sent_long["Sentiment"].map({
    "sent_neg": "Negative",
    "sent_neu": "Neutral",
    "sent_pos": "Positive"
})

fig = px.bar(
    sent_long,
    x="Mean Probability",
    y="Topic_Label_Wrapped",
    color="Sentiment",
    orientation="h",
    barmode="group",
    labels={
        "Mean Probability": "Mean Sentiment Probability",
        "Topic_Label_Wrapped": "Topic"
    },
    title="<b>Mean Sentiment Probabilities per Topic</b>"
)

fig.update_layout(
    width=1900,
    height=1100,
    margin=dict(l=460, r=160, t=140, b=100),
    legend_title="Sentiment"
)

fig.update_xaxes(range=[0, 1])
fig.update_yaxes(categoryorder="array", categoryarray=plot_df["Topic_Label_Wrapped"].tolist()[::-1])

fig = apply_thesis_style(
    fig,
    title_size=30,
    axis_title_size=24,
    tick_size=16,
    legend_size=18
)

save_plotly(
    fig,
    "sentiment_mean_probabilities_labeled",
    width=1900,
    height=1100
)

# ============================================================
# Figure: Sentiment Distribution per Topic
# ============================================================

sentiment_cols = [c for c in ["negative", "neutral", "positive"] if c in plot_df.columns]

if sentiment_cols:
    sent_dist_long = plot_df.melt(
        id_vars=["Topic", "Topic_Label", "Topic_Label_Wrapped"],
        value_vars=sentiment_cols,
        var_name="Sentiment",
        value_name="Share"
    )

    sent_dist_long["Sentiment"] = sent_dist_long["Sentiment"].str.capitalize()

    fig = px.bar(
        sent_dist_long,
        x="Share",
        y="Topic_Label_Wrapped",
        color="Sentiment",
        orientation="h",
        barmode="stack",
        labels={
            "Share": "Share of Posts",
            "Topic_Label_Wrapped": "Topic"
        },
        title="<b>Sentiment Distribution per Topic</b>"
    )

    fig.update_layout(
        width=1900,
        height=1100,
        margin=dict(l=460, r=160, t=140, b=100),
        legend_title="Sentiment"
    )

    fig.update_xaxes(range=[0, 1])
    fig.update_yaxes(categoryorder="array", categoryarray=plot_df["Topic_Label_Wrapped"].tolist()[::-1])

    fig = apply_thesis_style(
        fig,
        title_size=30,
        axis_title_size=24,
        tick_size=16,
        legend_size=18
    )

    save_plotly(
        fig,
        "sentiment_distribution_labeled",
        width=1900,
        height=1100
    )
else:
    print("No sentiment distribution columns found. Expected columns: negative, neutral, positive.")

# ============================================================
# Figure: Mean Emotion Scores per Topic
# ============================================================

possible_emotion_cols = [
    "sadness", "disgust", "anger", "pessimism", "fear",
    "anticipation", "joy", "optimism", "surprise", "love", "trust"
]

emotion_cols = [c for c in possible_emotion_cols if c in plot_df.columns]

print("Emotion columns used:", emotion_cols)

emo_plot = plot_df[["Topic_Label"] + emotion_cols].copy()
emo_plot["Topic_Label_Wrapped"] = wrap_series(emo_plot["Topic_Label"], width=36)
emo_plot = emo_plot.drop(columns=["Topic_Label"])
emo_plot = emo_plot.set_index("Topic_Label_Wrapped")

fig = px.imshow(
    emo_plot,
    labels=dict(
        x="Emotion",
        y="Topic",
        color="Mean Score"
    ),
    title="<b>Mean Emotion Scores per Topic</b>",
    aspect="auto"
)

fig.update_layout(
    width=1600,
    height=1100,
    margin=dict(l=500, r=120, t=140, b=140)
)

fig.update_xaxes(tickangle=45, automargin=True)
fig.update_yaxes(automargin=True)

fig = apply_thesis_style(
    fig,
    title_size=30,
    axis_title_size=24,
    tick_size=16,
    legend_size=18
)

save_plotly(
    fig,
    "emotion_heatmap_topic_means_labeled",
    width=1600,
    height=1100
)

# ============================================================
# Figure: Dominant Emotion per Topic
# ============================================================

top_emo = plot_df[["Topic", "Topic_Label", "Topic_Label_Wrapped"] + emotion_cols].copy()

top_emo["Dominant Emotion"] = top_emo[emotion_cols].idxmax(axis=1)
top_emo["Mean Score"] = top_emo[emotion_cols].max(axis=1)

top_emo = top_emo.sort_values("Mean Score", ascending=True)

fig = px.bar(
    top_emo,
    x="Mean Score",
    y="Topic_Label_Wrapped",
    color="Dominant Emotion",
    orientation="h",
    labels={
        "Mean Score": "Mean Score",
        "Topic_Label_Wrapped": "Topic"
    },
    title="<b>Dominant Emotion per Topic</b>"
)

fig.update_layout(
    width=1900,
    height=1000,
    margin=dict(l=460, r=160, t=140, b=100),
    legend_title="Dominant Emotion"
)

fig.update_xaxes(range=[0, max(0.85, top_emo["Mean Score"].max() * 1.10)])
fig.update_yaxes(categoryorder="array", categoryarray=top_emo["Topic_Label_Wrapped"])

fig = apply_thesis_style(
    fig,
    title_size=30,
    axis_title_size=24,
    tick_size=16,
    legend_size=18
)

save_plotly(
    fig,
    "emotion_top_emotion_per_topic_labeled",
    width=1900,
    height=1000
)

# ============================================================
# Figure: Emotion Correlation Heatmap
# ============================================================

emo_corr = rep_df[emotion_cols].corr()

emo_corr.to_csv(tables_dir / "emotion_correlation_matrix.csv")
emo_corr.to_excel(tables_dir / "emotion_correlation_matrix.xlsx")

print("Emotion correlation matrix saved to:", tables_dir)
display(emo_corr)

fig = px.imshow(
    emo_corr,
    labels=dict(
        x="Emotion",
        y="Emotion",
        color="Correlation"
    ),
    title="<b>Emotion Correlation Heatmap</b>",
    aspect="auto",
    zmin=-1,
    zmax=1
)

fig.update_layout(
    width=1200,
    height=1100,
    margin=dict(l=160, r=120, t=140, b=160)
)

fig.update_xaxes(tickangle=45, automargin=True)
fig.update_yaxes(automargin=True)

fig = apply_thesis_style(
    fig,
    title_size=30,
    axis_title_size=24,
    tick_size=18,
    legend_size=18
)

save_plotly(
    fig,
    "emotion_correlation_heatmap",
    width=1200,
    height=1100
)

print("All figures saved to:", figures_dir)

